# carregando em um dataframe e fazendo testes

importações

In [ ]:
# certifiquese de ter os pacotes instalados
!pip install transformers torch seaborn matplotlib spacy
!python -m spacy download pt_core_news_sm
!python -m spacy download en_core_web_sm

In [ ]:
import pandas as pd

nome_arquivo = 'DESTAQUES DE PUBLICAÇÕES.xlsx'

df = pd.read_excel(nome_arquivo, sheet_name='DESTAQUES')

print("--- Primeiras 5 linhas da aba DESTAQUES ---")
display(df.head()) 
print("--- ultimas 5 linhas da aba DESTAQUES ---")
display(df.tail()) 

periodicos_unicos = df['PERIÓDICO'].dropna().unique()

print(f"\nTotal de periódicos únicos para buscar no site: {len(periodicos_unicos)}")
print("\n--- Exemplos dos primeiros periódicos que vamos pesquisar ---")
for p in periodicos_unicos[:5]:
    print(f"- {p}")

# Testando as requisições ao site

In [ ]:
import requests
from bs4 import BeautifulSoup


periodico_teste = periodicos_unicos[0] 
print(f"Iniciando teste de busca para: {periodico_teste}\n")

url = "https://periodicos-adm.com/"

parametros = {'search_term': periodico_teste}

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

response = requests.get(url, params=parametros, headers=headers)

if response.status_code == 200:
    print("✅ Conexão com o site bem-sucedida!")
    

    soup = BeautifulSoup(response.text, 'html.parser')
   
    resultados = soup.find_all('div', class_='journal-card')
    
    if resultados:
        print(f"✅ Encontramos {len(resultados)} cartão(ões) de resultado para este periódico!\n")
        print("--- Estrutura HTML extraída do site ---")

        print(resultados[0].prettify()[:800]) 
    else:
        print("")
else:
    print(f"")

# iniciando a limpeza via re

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import spacy
import pandas as pd


try:
    nlp = spacy.load("pt_core_news_sm")
except:
    import os
    os.system("python -m spacy download pt_core_news_sm")
    nlp = spacy.load("pt_core_news_sm")

# --- MÓDULO DE PROCESSAMENTO (NLP + RE) ---

def processar_periodico_nlp(nome_original):
    """
    Combina Expressões Regulares (Etapa 1) com SpaCy (Etapa 1) 
    para preparar o dado para o Scraping (Etapa 2).
    """
    # [RE] Limpeza inicial: Remove parênteses e ruídos
    nome_limpo_re = re.sub(r'\(.*?\)', '', str(nome_original)).strip()
    
    # [SpaCy] Análise Linguística Profunda
    doc = nlp(nome_limpo_re)
    
    # Extraindo Lemas (radicais) e filtrando apenas o que importa (Substantivos e Adjetivos)
    # Removemos pontuação e stopwords (o, de, com, and, of)
    tokens_interessantes = [
        token.lemma_.lower() for token in doc 
        if not token.is_stop and not token.is_punct and token.pos_ in ['NOUN', 'ADJ', 'PROPN']
    ]
    
    # Criamos uma "assinatura temática" do periódico
    assinatura = " ".join(tokens_interessantes)
    
    return nome_limpo_re, assinatura

# --- EXECUÇÃO E SCRAPING 
# Exemplo pegando o primeiro da sua lista
periodico_raw = periodicos_unicos[0] 

# Aplicando nossa função de NLP
nome_para_busca, termos_chave = processar_periodico_nlp(periodico_raw)

print(f" Nome Original: {periodico_raw}")
print(f"Nome Limpo (RE): {nome_para_busca}")
print(f"Termos Chave (SpaCy): {termos_chave}\n")

# Configuração da requisição (Web Scraping - Etapa 2)
url = "https://periodicos-adm.com/"
parametros = {'search_term': nome_para_busca} 
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

print(f" Iniciando busca no site para: {nome_para_busca}...")
response = requests.get(url, params=parametros, headers=headers)

if response.status_code == 200:
    print(" Conexão com o site bem-sucedida!")
    soup = BeautifulSoup(response.text, 'html.parser')
    resultados = soup.find_all('div', class_='journal-card')
    
    if resultados:
        print(f"Sucesso! Encontramos {len(resultados)} resultado(s)!\n")
        
        # Exemplo de extração de dados não estruturados do primeiro card
        primeiro_resultado = resultados[0]
        titulo_site = primeiro_resultado.find('h2').get_text(strip=True) if primeiro_resultado.find('h2') else "N/A"
        
        print(f"Título encontrado no site: {titulo_site}")
        print("--- Estrutura HTML extraída ---")
        print(primeiro_resultado.prettify()[:500]) 
    else:
        print(f"")
else:
    print(f"")

In [ ]:
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

def processamento_avancado_spacy(texto):
    if not isinstance(texto, str):
        return {}
    
    doc = nlp(texto)
    
    # 1. Segmentação de Sentenças
    sentencas = [sent.text for sent in doc.sents]
    
    # 2. Tokenização, Normalização (minúsculas), Lematização e POS Tagging
    tokens_info = []
    for token in doc:
        if not token.is_stop and not token.is_punct: # Removendo stopwords e pontuação
            tokens_info.append({
                'Original': token.text,
                'Normalizado': token.lower_,
                'Lema': token.lemma_,
                'POS': token.pos_
            })
            
    # 3. Noun Chunks (Fragmentos nominais)
    noun_chunks = [chunk.text for chunk in doc.noun_chunks]
    
    # 4. Reconhecimento de Entidades Nomeadas (NER)
    entidades = [(ent.text, ent.label_) for ent in doc.ents]
    
    return {
        'Qtd_Sentencas': len(sentencas),
        'Tokens_Relevantes': len(tokens_info),
        'Noun_Chunks': noun_chunks,
        'Entidades': entidades
    }

print("Aplicando pipeline do spaCy nos títulos das produções...")
# Aplicar a função apenas nos primeiros 10 registros para demonstração rápida, ou no df todo
df['Analise_spaCy'] = df['TÍTULO PRODUÇÃO'].apply(processamento_avancado_spacy)

# Mostrando o resultado de uma linha específica para ver o detalhamento
exemplo_spacy = df.iloc[66]['Analise_spaCy'] # O título: "THE OVER-CONCENTRATION OF INNOVATION..."
print("\n--- Resultado spaCy para o artigo 66 ---")
print(f"Noun Chunks: {exemplo_spacy.get('Noun_Chunks')}")
print(f"Entidades Encontradas: {exemplo_spacy.get('Entidades')}")

In [ ]:
from transformers import pipeline
import torch

print("Carregando modelos baseados em Transformers (isso pode levar alguns instantes)...")

# 1. Pipeline de Análise de Sentimentos (Multilíngue)
sentiment_analyzer = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

# 2. Pipeline de Classificação Zero-Shot
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
categorias_tematicas = ["Technology and Innovation", "Business and Management", "Public Policy", "Sustainability"]

def analisar_sentimento_transformer(texto):
    if not isinstance(texto, str):
        return "Neutro"
    try:
        resultado = sentiment_analyzer(texto[:512])[0] # Limite de 512 tokens do BERT
        label = resultado['label']
        if label in ['1 star', '2 stars']: return 'Negativo'
        elif label == '3 stars': return 'Neutro'
        else: return 'Positivo'
    except:
        return "Erro"

def classificar_tema_transformer(texto):
    if not isinstance(texto, str):
        return "Desconhecido"
    try:
        resultado = zero_shot_classifier(texto[:512], categorias_tematicas)
        return resultado['labels'][0] 
    except:
        return "Erro"

print("Executando inferência de sentimentos...")
df['Sentimento'] = df['TÍTULO PRODUÇÃO'].apply(analisar_sentimento_transformer)

print("Executando inferência de classificação temática...")
df['Categoria_Tematica'] = df['TÍTULO PRODUÇÃO'].apply(classificar_tema_transformer)

display(df[['TÍTULO PRODUÇÃO', 'Sentimento', 'Categoria_Tematica']].head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Análise de Sentimentos
sns.countplot(data=df, x='Sentimento', palette='viridis', ax=axes[0])
axes[0].set_title('Distribuição de Sentimentos nos Títulos das Produções', fontsize=14)
axes[0].set_xlabel('Sentimento', fontsize=12)
axes[0].set_ylabel('Quantidade de Artigos', fontsize=12)

# Gráfico 2: Classificação Temática Zero-Shot
sns.countplot(data=df, y='Categoria_Tematica', palette='magma', order=df['Categoria_Tematica'].value_counts().index, ax=axes[1])
axes[1].set_title('Classificação Temática (Zero-Shot via Transformers)', fontsize=14)
axes[1].set_xlabel('Quantidade de Artigos', fontsize=12)
axes[1].set_ylabel('Categoria', fontsize=12)

plt.tight_layout()
plt.show()